In [1]:
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Flatten, Dense, BatchNormalization
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# Load Dataset
X_train_raw = np.loadtxt('DataSet/Train/X_train.txt')
y_train_raw = np.loadtxt('DataSet/Train/y_train.txt').astype(int) - 1  # 将标签转换为 0 索引

X_test = np.loadtxt('DataSet/Test/X_test.txt')
y_test = np.loadtxt('DataSet/Test/y_test.txt').astype(int) - 1



In [5]:
# Encode labels from 1–12 → 0–11
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_raw)
y_test_encoded = le.transform(y_test)

In [8]:
# Balancing the data with SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_raw, y_train_encoded)


In [9]:
# Reshape for CNN input 
X_train_cnn = X_train_balanced.reshape(X_train_balanced.shape[0], X_train_balanced.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)


In [10]:
# 3. One-hot encode the labels
y_train_balanced_cat = to_categorical(y_train_balanced)
y_test_cat = to_categorical(y_test_encoded)

In [11]:
# Build CNN model
model = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Conv1D(128, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(y_train_balanced_cat.shape[1], activation='softmax')
])


In [12]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [13]:
# Train the model
model.fit(X_train_cnn, y_train_balanced_cat, epochs=30, batch_size=64, validation_data=(X_test_cnn, y_test_cat))


Epoch 1/30
267/267 [==============================] - 65s 239ms/step - loss: 0.8277 - accuracy: 0.7089 - val_loss: 6.3183 - val_accuracy: 0.1509
Epoch 2/30
267/267 [==============================] - 62s 232ms/step - loss: 0.2884 - accuracy: 0.8822 - val_loss: 0.2982 - val_accuracy: 0.9045
Epoch 3/30
267/267 [==============================] - 62s 233ms/step - loss: 0.2023 - accuracy: 0.9177 - val_loss: 0.2259 - val_accuracy: 0.9257
Epoch 4/30
267/267 [==============================] - 61s 227ms/step - loss: 0.1718 - accuracy: 0.9285 - val_loss: 0.2708 - val_accuracy: 0.9336
Epoch 5/30
267/267 [==============================] - 61s 227ms/step - loss: 0.1424 - accuracy: 0.9425 - val_loss: 0.2746 - val_accuracy: 0.9288
Epoch 6/30
267/267 [==============================] - 61s 227ms/step - loss: 0.1192 - accuracy: 0.9506 - val_loss: 0.2479 - val_accuracy: 0.9342
Epoch 7/30
267/267 [==============================] - 61s 227ms/step - loss: 0.1081 - accuracy: 0.9574 - val_loss: 0.2865 - val_ac

In [4]:
#Evaluate the model
y_pred = model.predict(X_test_cnn)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

loss, acc = model.evaluate(X_test_cnn, y_test_cat)
print(f"Test Accuracy: {acc:.4f}")

print("Classification Report:\n", classification_report(y_true, y_pred_classes))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred_classes))

# Evaluate on test set
loss, acc = model.evaluate(X_test_cnn, y_test_cat)
print(f"Test Accuracy: {acc:.4f}")



NameError: name 'model' is not defined